## 1. Reading in the data
We have to first read the data from the *data* directory. We will use -9999 for missing values and convert them to N/A. Additionally, we will use whitespace as the delimiter. 

In [69]:
# All imports needed in this notebook in one cell to simplify importing
import pandas as pd
from math import isnan

In [70]:
# Define path for the data file
file = r'data/2315676.txt'
# Read the date using whitespace as the delimiter, -9999 as missing values, 0th row as the header and
# additionally skip 1st row where there is just dashes
data = pd.read_csv(file, delim_whitespace=True, na_values=[-9999], header=[0], skiprows=[1])
# Print data
print(data)
# Get the number of original data rows
orig_n = len(data)

                 STATION      STATION_NAME      DATE  TAVG  TMAX  TMIN
0      GHCND:FI000007501  SODANKYLA-AWS-FI  19080101   NaN   2.0 -37.0
1      GHCND:FI000007501  SODANKYLA-AWS-FI  19080102   NaN   6.0 -26.0
2      GHCND:FI000007501  SODANKYLA-AWS-FI  19080103   NaN   7.0 -27.0
3      GHCND:FI000007501  SODANKYLA-AWS-FI  19080104   NaN  -3.0 -27.0
4      GHCND:FI000007501  SODANKYLA-AWS-FI  19080105   NaN   4.0 -36.0
...                  ...               ...       ...   ...   ...   ...
41060  GHCND:FI000007501  SODANKYLA-AWS-FI  20201003  47.0  51.0   NaN
41061  GHCND:FI000007501  SODANKYLA-AWS-FI  20201004  43.0  47.0  37.0
41062  GHCND:FI000007501  SODANKYLA-AWS-FI  20201005  42.0   NaN  37.0
41063  GHCND:FI000007501  SODANKYLA-AWS-FI  20201006  45.0  46.0  43.0
41064  GHCND:FI000007501  SODANKYLA-AWS-FI  20201007  47.0  50.0  44.0

[41065 rows x 6 columns]


## 2. Calculating estimated temperatures
We can immediately see that there are some missing values in the TAVG column. For those columns, we can estimate an average by averaging the TMAX and TMIN values. However, there are some rows where either of those values is missing. For rows where either of the TMAX and TMIN values is missing, the return value for the calculation will also be N/A. 

In [71]:
# Calculate estimated average for all rows. 
data['ESTAVG'] = (data['TMAX'] + data['TMIN'])/2
# Take copy of the data for a future sanity check
data_copy = data.copy()
# Drop rows where both the TAVG and ESTAVG values are missing
data = data.dropna(axis=0, how='all', subset=['TAVG', 'ESTAVG'])
# Get the number of rows after dropping the N/A rows
fil_n = len(data)
print(data)

                 STATION      STATION_NAME      DATE  TAVG  TMAX  TMIN  ESTAVG
0      GHCND:FI000007501  SODANKYLA-AWS-FI  19080101   NaN   2.0 -37.0   -17.5
1      GHCND:FI000007501  SODANKYLA-AWS-FI  19080102   NaN   6.0 -26.0   -10.0
2      GHCND:FI000007501  SODANKYLA-AWS-FI  19080103   NaN   7.0 -27.0   -10.0
3      GHCND:FI000007501  SODANKYLA-AWS-FI  19080104   NaN  -3.0 -27.0   -15.0
4      GHCND:FI000007501  SODANKYLA-AWS-FI  19080105   NaN   4.0 -36.0   -16.0
...                  ...               ...       ...   ...   ...   ...     ...
41060  GHCND:FI000007501  SODANKYLA-AWS-FI  20201003  47.0  51.0   NaN     NaN
41061  GHCND:FI000007501  SODANKYLA-AWS-FI  20201004  43.0  47.0  37.0    42.0
41062  GHCND:FI000007501  SODANKYLA-AWS-FI  20201005  42.0   NaN  37.0     NaN
41063  GHCND:FI000007501  SODANKYLA-AWS-FI  20201006  45.0  46.0  43.0    44.5
41064  GHCND:FI000007501  SODANKYLA-AWS-FI  20201007  47.0  50.0  44.0    47.0

[40882 rows x 7 columns]


We can see that the number of rows we filtered out is

In [72]:
print(orig_n - fil_n)

183


We can confirm that the number of dropped rows make sense by selecting those rows from the *data* in a different way. We will select the rows where both of those are missing. Then as we use *len()* for the selection we should have 183 rows. Let's do it! 

In [73]:
# Select rows where both TAVG and ESTAVG are missing
len(data_copy.loc[(data_copy['TAVG'].isna()) & (data_copy['ESTAVG'].isna())])

183

which confirms, that the *dropna* worked as it should.

## 3. Temperature unit conversion
In the original data, all the temperatures are given in Fahrenheit so we have to convert them to Celsius using the function we'll define here next.

In [74]:
def fahr_to_celsius(temp_fahrenheit):
    """
    Converts temperature values given in Fahrenheit (param: temp_fahrenheit) to Celsius degrees.
    Usage: fahr_to_celsius(48)
    """
    converted_temp = (temp_fahrenheit - 32) / 1.8
    return converted_temp

Then we can use the function for the average temperature columns. Let's define new columns for them with the prefix *C_*:

In [75]:
# Convert AVG to Celsius
data['C_TAVG'] = data['TAVG'].apply(fahr_to_celsius)
# Convert ESTAVG to Celsius
data['C_ESTAVG'] = data['ESTAVG'].apply(fahr_to_celsius)
#print(data[''])

Then we have to combine the two Series for average temperature favoring the actual average over the estimated. Reading the documentation of Pandas it seems *Pandas.Series.combine_first()* does absolutely that. Let' use it!

In [76]:
# Create a Series for combined average temperatures
data['C_COMB_AVG'] = data['C_TAVG'].combine_first(data['C_ESTAVG'])

In [77]:
# Print out the average temperature columns
data[['C_TAVG', 'C_ESTAVG', 'C_COMB_AVG']]

,C_TAVG,C_ESTAVG,C_COMB_AVG
0,NaN,-27.500000,-27.500000
1,NaN,-23.333333,-23.333333
2,NaN,-23.333333,-23.333333
3,NaN,-26.111111,-26.111111
4,NaN,-26.666667,-26.666667
...,...,...,...
41060,8.333333,NaN,8.333333
41061,6.111111,5.555556,6.111111
41062,5.555556,NaN,5.555556
41063,7.222222,6.944444,7.222222


It now seems we got just what we wanted, as it seems like C_TAVG is favored over C_ESTAVG. Let's check if there are any NaN left. There shouldn't. 

In [78]:
# Count the number of NaN
data['C_COMB_AVG'].isna().sum()

0

As expected, there are no NaN values left so we can continue to the next step.

## 4. Calculating seasonal average temperatures
As we look into the task of calculating the seasonal average temperatures, we notice that the *winter* season begins in December and ends after February. Therefore, it's evident the winter season takes place during two years, although we have to confine it two one year to have the analysis make sense. We define new term *seasonal year*, label it $s\_year$, which defines in which yearly season the monthly temperature belongs. In this case, we define that December always belongs to the next years winter season rather than the current year. We could also use the months Jan, Feb and December but somehow the analysis seems more meaningful if there are no gaps between the months. For example, by this definition, 2021-12-1 would get assigned $s\_year = 2022$ and every other month would get the actual year as the seasonal year. In essence, the winter season of 1909 begins, by definition, in 1908.

In [79]:
# First, we need to get the actual year and month from the DATE column
data['YEAR'] = data['DATE'].astype(str).str.slice(start=0, stop=4).astype(int)
data['MONTH'] = data['DATE'].astype(str).str.slice(start=4, stop=6).astype(int)
# Print the date, year and month
print(data[['DATE', 'YEAR', 'MONTH']])
# Let's assign seasonal year and add it as a column to the data

           DATE  YEAR  MONTH
0      19080101  1908      1
1      19080102  1908      1
2      19080103  1908      1
3      19080104  1908      1
4      19080105  1908      1
...         ...   ...    ...
41060  20201003  2020     10
41061  20201004  2020     10
41062  20201005  2020     10
41063  20201006  2020     10
41064  20201007  2020     10

[40882 rows x 3 columns]


In [80]:
# Let's assign seasonal year and add it as a column to the dataframe. 
# Assign first according to the usual year but fix later for December
data['S_YEAR'] = data['YEAR']
# Now fix December
data.loc[data['MONTH'] == 12, 'S_YEAR'] = data['S_YEAR'] + 1

In [81]:
# Let's print the dates to see that the assignment looks OK
print(data[['DATE', 'YEAR', 'MONTH', 'S_YEAR']].loc[data['MONTH'] == 12])

           DATE  YEAR  MONTH  S_YEAR
335    19081201  1908     12    1909
336    19081202  1908     12    1909
337    19081203  1908     12    1909
338    19081204  1908     12    1909
339    19081205  1908     12    1909
...         ...   ...    ...     ...
40780  20191227  2019     12    2020
40781  20191228  2019     12    2020
40782  20191229  2019     12    2020
40783  20191230  2019     12    2020
40784  20191231  2019     12    2020

[3445 rows x 4 columns]


In [82]:
# Then also all the date data, not just for December
print(data[['DATE', 'YEAR', 'MONTH', 'S_YEAR']])

           DATE  YEAR  MONTH  S_YEAR
0      19080101  1908      1    1908
1      19080102  1908      1    1908
2      19080103  1908      1    1908
3      19080104  1908      1    1908
4      19080105  1908      1    1908
...         ...   ...    ...     ...
41060  20201003  2020     10    2020
41061  20201004  2020     10    2020
41062  20201005  2020     10    2020
41063  20201006  2020     10    2020
41064  20201007  2020     10    2020

[40882 rows x 4 columns]


Both seem to be following the definition we gave, so we can go forward with it. Let's assign the seasons next.

In [83]:
# Let's start by assigning something to the column SEASON
data['SEASON'] = ""
# Then we will select and assign each season separately
data.loc[data['MONTH'].isin([12,1,2]), 'SEASON'] = 'WINTER'
# Then select those months to check that the assignment worked correctly and print
print(data.loc[data['MONTH'].isin([1,2,12]), ['MONTH', 'SEASON']])
print(data.loc[data['MONTH'].isin([12]), ['MONTH', 'SEASON']])

       MONTH  SEASON
0          1  WINTER
1          1  WINTER
2          1  WINTER
3          1  WINTER
4          1  WINTER
...      ...     ...
40840      2  WINTER
40841      2  WINTER
40842      2  WINTER
40843      2  WINTER
40844      2  WINTER

[10093 rows x 2 columns]
       MONTH  SEASON
335       12  WINTER
336       12  WINTER
337       12  WINTER
338       12  WINTER
339       12  WINTER
...      ...     ...
40780     12  WINTER
40781     12  WINTER
40782     12  WINTER
40783     12  WINTER
40784     12  WINTER

[3445 rows x 2 columns]


In [84]:
# Then do the rest of the seasons
data.loc[data['MONTH'].isin([3,4,5]), 'SEASON'] = 'SPRING'
# Then select those months to check that the assignment worked correctly and print
print(data.loc[data['MONTH'].isin([3,4,5]), ['MONTH', 'SEASON']])
print(data.loc[data['MONTH'].isin([4]), ['MONTH', 'SEASON']])

       MONTH  SEASON
60         3  SPRING
61         3  SPRING
63         3  SPRING
64         3  SPRING
65         3  SPRING
...      ...     ...
40932      5  SPRING
40933      5  SPRING
40934      5  SPRING
40935      5  SPRING
40936      5  SPRING

[10308 rows x 2 columns]
       MONTH  SEASON
91         4  SPRING
92         4  SPRING
93         4  SPRING
94         4  SPRING
95         4  SPRING
...      ...     ...
40901      4  SPRING
40902      4  SPRING
40903      4  SPRING
40904      4  SPRING
40905      4  SPRING

[3338 rows x 2 columns]


In [85]:
data.loc[data['MONTH'].isin([6,7,8]), 'SEASON'] = 'SUMMER'
# Then select those months to check that the assignment worked correctly and print
print(data.loc[data['MONTH'].isin([6,7,8]), ['MONTH', 'SEASON']])
print(data.loc[data['MONTH'].isin([7]), ['MONTH', 'SEASON']])

       MONTH  SEASON
152        6  SUMMER
153        6  SUMMER
154        6  SUMMER
155        6  SUMMER
156        6  SUMMER
...      ...     ...
41023      8  SUMMER
41024      8  SUMMER
41025      8  SUMMER
41026      8  SUMMER
41027      8  SUMMER

[10385 rows x 2 columns]
       MONTH  SEASON
182        7  SUMMER
183        7  SUMMER
184        7  SUMMER
185        7  SUMMER
186        7  SUMMER
...      ...     ...
40993      7  SUMMER
40994      7  SUMMER
40995      7  SUMMER
40996      7  SUMMER
40997      7  SUMMER

[3495 rows x 2 columns]


In [86]:
data.loc[data['MONTH'].isin([9,10,11]), 'SEASON'] = 'AUTUMN'
# Then select those months to check that the assignment worked correctly and print
print(data.loc[data['MONTH'].isin([9,10,11]), ['MONTH', 'SEASON']])
print(data.loc[data['MONTH'].isin([10]), ['MONTH', 'SEASON']])

       MONTH  SEASON
244        9  AUTUMN
245        9  AUTUMN
246        9  AUTUMN
247        9  AUTUMN
248        9  AUTUMN
...      ...     ...
41060     10  AUTUMN
41061     10  AUTUMN
41062     10  AUTUMN
41063     10  AUTUMN
41064     10  AUTUMN

[10096 rows x 2 columns]
       MONTH  SEASON
274       10  AUTUMN
275       10  AUTUMN
276       10  AUTUMN
277       10  AUTUMN
278       10  AUTUMN
...      ...     ...
41060     10  AUTUMN
41061     10  AUTUMN
41062     10  AUTUMN
41063     10  AUTUMN
41064     10  AUTUMN

[3439 rows x 2 columns]


Let's print all the useful columns for change for all months just as a sanity check

In [87]:
data.loc[data['MONTH'].isin([1,2]), ['STATION', 'YEAR', 'MONTH', 'SEASON', 'S_YEAR', 'C_COMB_AVG']]

,STATION,YEAR,MONTH,SEASON,S_YEAR,C_COMB_AVG
0,GHCND:FI000007501,1908,1,WINTER,1908,-27.500000
1,GHCND:FI000007501,1908,1,WINTER,1908,-23.333333
2,GHCND:FI000007501,1908,1,WINTER,1908,-23.333333
3,GHCND:FI000007501,1908,1,WINTER,1908,-26.111111
4,GHCND:FI000007501,1908,1,WINTER,1908,-26.666667
...,...,...,...,...,...,...
40840,GHCND:FI000007501,2020,2,WINTER,2020,-13.333333
40841,GHCND:FI000007501,2020,2,WINTER,2020,-18.888889
40842,GHCND:FI000007501,2020,2,WINTER,2020,-11.666667
40843,GHCND:FI000007501,2020,2,WINTER,2020,-10.555556


In [88]:
data.loc[data['MONTH'].isin([3,4]), ['STATION', 'YEAR', 'MONTH', 'SEASON', 'S_YEAR', 'C_COMB_AVG']]

,STATION,YEAR,MONTH,SEASON,S_YEAR,C_COMB_AVG
60,GHCND:FI000007501,1908,3,SPRING,1908,-4.722222
61,GHCND:FI000007501,1908,3,SPRING,1908,-6.944444
63,GHCND:FI000007501,1908,3,SPRING,1908,-8.888889
64,GHCND:FI000007501,1908,3,SPRING,1908,-12.777778
65,GHCND:FI000007501,1908,3,SPRING,1908,-3.333333
...,...,...,...,...,...,...
40901,GHCND:FI000007501,2020,4,SPRING,2020,0.555556
40902,GHCND:FI000007501,2020,4,SPRING,2020,-1.111111
40903,GHCND:FI000007501,2020,4,SPRING,2020,-1.111111
40904,GHCND:FI000007501,2020,4,SPRING,2020,-1.666667


In [89]:
data.loc[data['MONTH'].isin([5,6]), ['STATION', 'YEAR', 'MONTH', 'SEASON', 'S_YEAR', 'C_COMB_AVG']]

,STATION,YEAR,MONTH,SEASON,S_YEAR,C_COMB_AVG
122,GHCND:FI000007501,1908,5,SPRING,1908,0.000000
123,GHCND:FI000007501,1908,5,SPRING,1908,-2.222222
124,GHCND:FI000007501,1908,5,SPRING,1908,-2.777778
125,GHCND:FI000007501,1908,5,SPRING,1908,-2.500000
126,GHCND:FI000007501,1908,5,SPRING,1908,-2.777778
...,...,...,...,...,...,...
40962,GHCND:FI000007501,2020,6,SUMMER,2020,16.666667
40963,GHCND:FI000007501,2020,6,SUMMER,2020,11.666667
40964,GHCND:FI000007501,2020,6,SUMMER,2020,15.000000
40965,GHCND:FI000007501,2020,6,SUMMER,2020,17.222222


In [90]:
data.loc[data['MONTH'].isin([7,8]), ['STATION', 'YEAR', 'MONTH', 'SEASON', 'S_YEAR', 'C_COMB_AVG']]

,STATION,YEAR,MONTH,SEASON,S_YEAR,C_COMB_AVG
182,GHCND:FI000007501,1908,7,SUMMER,1908,9.166667
183,GHCND:FI000007501,1908,7,SUMMER,1908,6.666667
184,GHCND:FI000007501,1908,7,SUMMER,1908,4.722222
185,GHCND:FI000007501,1908,7,SUMMER,1908,5.277778
186,GHCND:FI000007501,1908,7,SUMMER,1908,9.444444
...,...,...,...,...,...,...
41023,GHCND:FI000007501,2020,8,SUMMER,2020,7.222222
41024,GHCND:FI000007501,2020,8,SUMMER,2020,7.777778
41025,GHCND:FI000007501,2020,8,SUMMER,2020,10.000000
41026,GHCND:FI000007501,2020,8,SUMMER,2020,10.555556


In [91]:
data.loc[data['MONTH'].isin([9,10]), ['STATION', 'YEAR', 'MONTH', 'SEASON', 'S_YEAR', 'C_COMB_AVG']]

,STATION,YEAR,MONTH,SEASON,S_YEAR,C_COMB_AVG
244,GHCND:FI000007501,1908,9,AUTUMN,1908,7.777778
245,GHCND:FI000007501,1908,9,AUTUMN,1908,10.555556
246,GHCND:FI000007501,1908,9,AUTUMN,1908,6.666667
247,GHCND:FI000007501,1908,9,AUTUMN,1908,9.444444
248,GHCND:FI000007501,1908,9,AUTUMN,1908,4.722222
...,...,...,...,...,...,...
41060,GHCND:FI000007501,2020,10,AUTUMN,2020,8.333333
41061,GHCND:FI000007501,2020,10,AUTUMN,2020,6.111111
41062,GHCND:FI000007501,2020,10,AUTUMN,2020,5.555556
41063,GHCND:FI000007501,2020,10,AUTUMN,2020,7.222222


In [92]:
data.loc[data['MONTH'].isin([11,12]), ['STATION', 'YEAR', 'MONTH', 'SEASON', 'S_YEAR', 'C_COMB_AVG']]

,STATION,YEAR,MONTH,SEASON,S_YEAR,C_COMB_AVG
305,GHCND:FI000007501,1908,11,AUTUMN,1908,0.555556
306,GHCND:FI000007501,1908,11,AUTUMN,1908,-1.666667
307,GHCND:FI000007501,1908,11,AUTUMN,1908,-7.222222
308,GHCND:FI000007501,1908,11,AUTUMN,1908,-9.166667
309,GHCND:FI000007501,1908,11,AUTUMN,1908,-5.555556
...,...,...,...,...,...,...
40780,GHCND:FI000007501,2019,12,WINTER,2020,-19.444444
40781,GHCND:FI000007501,2019,12,WINTER,2020,-8.333333
40782,GHCND:FI000007501,2019,12,WINTER,2020,-0.555556
40783,GHCND:FI000007501,2019,12,WINTER,2020,-1.666667


In [96]:
# Let's group the data by S_YEAR and SEASON in that order
grouped = data[['DATE', 'YEAR', 'MONTH', 'SEASON', 'S_YEAR', 'C_COMB_AVG']].groupby(['S_YEAR', 'SEASON'])
# Then aggregate the column of combined averages
year_seasonal_means = grouped.aggregate({'C_COMB_AVG': 'mean'})

In [97]:
# Print means as a sanity check
year_seasonal_means

C_COMB_AVG
S_YEAR SEASON            
1908   AUTUMN   -1.859217
       SPRING   -2.346743
       SUMMER   11.425926
       WINTER  -14.781746
1909   AUTUMN   -0.155827
...                   ...
2019   WINTER  -12.370370
2020   AUTUMN    7.717718
       SPRING   -0.827295
       SUMMER   14.438339
       WINTER   -7.973138

[452 rows x 1 columns]

In [3]:
data.columns

NameError: name 'data' is not defined

In [2]:
data['SYEAR_SEASON'] = data['S_YEAR'].to_string() + data['SEASON']

NameError: name 'data' is not defined

In [1]:
data['SYEAR_SEASON']

NameError: name 'data' is not defined